<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/demos/Week_14_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 14 demo: Causation and Uplift Modeling



## Setup and Data Loading

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/demos/customer_upgrade_data.csv'
df = pd.read_csv(url)

# Inspect the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Let's split the data into treatment and control groups.

In [ ]:
# Split into treatment and control groups
treatment = df[df['received_offer'] == 1]
control = df[df['received_offer'] == 0]

print(f"Treatment group: {len(treatment):,} customers")
print(f"Control group:   {len(control):,} customers")

## Calculate the Naive Average Treatment Effect (ATE)

In [ ]:
# Calculate upgrade rates for treatment and control
p1_naive = treatment['upgraded'].mean()
p0_naive = control['upgraded'].mean()

print(f"\nTreatment upgrade rate: {p1_naive:.3f} ({100*p1_naive:.1f}%)")
print(f"Control upgrade rate:   {p0_naive:.3f} ({100*p0_naive:.1f}%)")

# Naive ATE = Treatment rate - Control rate
naive_ate = p1_naive - p0_naive
print(f"Naive ATE: {naive_ate:.3f} ({100*naive_ate:.1f} percentage points)")

print(f"\nBusiness interpretation:")
print(f"  On average, the offer appears to increase upgrade probability by {100*naive_ate:.1f} pp")

**Compare to ground truth:** Since this is simulated data, we have the true average treatment effect. Let's see how far off the naive ATE is.

In [ ]:
url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/demos/customer_upgrade_data_with_ground_truth.csv'
df_gt = pd.read_csv(url)
true_ate = df_gt['treatment_effect'].mean()

print(f"Naive ATE:  {naive_ate*100:+.1f} pp  (what we'd report)")
print(f"True ATE:   {true_ate*100:+.1f} pp  (ground truth)")
print(f"Bias:       {(naive_ate - true_ate)*100:+.1f} pp  ({100*(naive_ate-true_ate)/true_ate:+.0f}% overestimate)")

## Check for selection bias

In a randomized A/B test, treatment and control groups should look alike on every pre-treatment feature. If they don't, there's selection bias.
We'll check using the **Standardized Mean Difference (SMD)**.

$$\text{SMD} = \frac{\bar{X}_T - \bar{X}_C}{\sqrt{(s_T^2 + s_C^2) / 2}}$$

When |SMD| > 0.1 is a common red flag

In [ ]:
import numpy as np

# Standardized mean differences across features
features = ['age', 'tenure_months', 'data_usage_gb', 'minutes_used',
            'app_usage_score', 'login_frequency', 'monthly_charges',
            'late_payments_12mo']

print(f"{'Feature':<22s} {'T mean':>9s} {'C mean':>9s} {'SMD':>8s}  {'Flag':>5s}")
print("-" * 62)
for feat in features:
    t_mean = treatment[feat].mean()
    c_mean = control[feat].mean()
    pooled_sd = np.sqrt((treatment[feat].var() + control[feat].var()) / 2)
    smd = (t_mean - c_mean) / pooled_sd if pooled_sd > 0 else 0
    flag = '***' if abs(smd) > 0.1 else ''
    print(f"{feat:<22s} {t_mean:>9.2f} {c_mean:>9.2f} {smd:>+8.3f}  {flag:>5s}")

print("\n*** = imbalance threshold exceeded (|SMD| > 0.1)")

## Propensity Score Matching

Let's try PSM on our data.

### Step 1: Choose the covariates to include in PSM

In [ ]:
# One-hot encode categorical features (matching only works on numerical data)
df_enc = pd.get_dummies(
    df, columns=['contract_type', 'payment_method', 'gender', 'state'],
    drop_first=True
)

# Select features for the propensity model (exclude ID, outcome, treatment indicator)
exclude = ['customer_id', 'current_plan', 'upgraded', 'received_offer']
feature_cols = [c for c in df_enc.columns if c not in exclude]

X = df_enc[feature_cols].astype(float)
T = df_enc['received_offer'].values

print(f"Customers: {len(df):,}  |  Features: {len(feature_cols)}")
print(f"Treatment: {T.sum():,}  |  Control: {(1-T).sum():,}")

### Step 2: Use logistic regression to fit a PSM model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X_scaled, T)

# Estimated PS for each customer
df_enc['propensity'] = ps_model.predict_proba(X_scaled)[:, 1]

# AUC diagnostic: if treatment were random, AUC would be ~0.5
auc = roc_auc_score(T, df_enc['propensity'])
print(f"Propensity model AUC: {auc:.3f}")
print(f"(AUC = 0.5 means random assignment. Higher means selection bias is present.)")

### Step 3: Match treated customers to their nearest control neighbor by propensity

For each treated customer, find the control customer with the closest propensity score.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Get the treatment and control groups from the one-hot encoded dataset
treatment_enc = df_enc[df_enc['received_offer'] == 1]
control_enc = df_enc[df_enc['received_offer'] == 0]

# Propensity scores as 1-column arrays
T_ps = treatment_enc['propensity'].values.reshape(-1, 1)
C_ps = control_enc['propensity'].values.reshape(-1, 1)

# Create a nearest neighbors model from the control propensity scores
# Note: n_neighbors = 1 finds only one match
nn = NearestNeighbors(n_neighbors=1)
nn.fit(C_ps)

# Find the matches of the treatment data
distances, indices = nn.kneighbors(T_ps)

# Determine the matched treatment and control data, ensuring 'propensity' column is included
treatment_matched = treatment_enc.copy()
control_matched = control_enc.iloc[indices.flatten()]

Let's confirm that the matching results in near-identical PS distributions for treatment and control....

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for plotting
propensity_before = pd.DataFrame({
    'Propensity Score': pd.concat([treatment_enc['propensity'], control_enc['propensity']]),
    'Group': ['Treatment (Before)'] * len(treatment_enc) + ['Control (Before)'] * len(control_enc)
})

propensity_after = pd.DataFrame({
    'Propensity Score': pd.concat([treatment_matched['propensity'], control_matched['propensity']]),
    'Group': ['Treatment (After)'] * len(treatment_matched) + ['Control (After)'] * len(control_matched)
})

# Create the plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

sns.histplot(data=propensity_before, x='Propensity Score', hue='Group', kde=True, ax=axes[0], palette='coolwarm')
axes[0].set_title('Propensity Score Distribution Before Matching')
axes[0].set_ylabel('Density')

sns.histplot(data=propensity_after, x='Propensity Score', hue='Group', kde=True, ax=axes[1], palette='coolwarm')
axes[1].set_title('Propensity Score Distribution After Matching')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()

### Step 4: Check balance of covariates after matching

Now, let's re-check the Standardized Mean Difference (SMD) for the features using our matched groups. We expect to see a significant reduction in SMDs, ideally below the 0.1 threshold.

In [ ]:
print(f"{'Feature':<22s} {'T mean':>9s} {'C mean':>9s} {'SMD':>8s}  {'Flag':>5s}")
print("-" * 62)
for feat in features:
    t_mean_matched = treatment_matched[feat].mean()
    c_mean_matched = control_matched[feat].mean()
    # Pooled standard deviation of the matched groups
    pooled_sd_matched = np.sqrt((treatment_matched[feat].var() + control_matched[feat].var()) / 2)
    smd_matched = (t_mean_matched - c_mean_matched) / pooled_sd_matched if pooled_sd_matched > 0 else 0
    flag_matched = '***' if abs(smd_matched) > 0.1 else ''
    print(f"{feat:<22s} {t_mean_matched:>9.2f} {c_mean_matched:>9.2f} {smd_matched:>+8.3f}  {flag_matched:>5s}")

print("\n*** = imbalance threshold exceeded (|SMD| > 0.1)")

print("\nComparison with original SMDs:")
print(f"{'Feature':<22s} {'Original SMD':>14s} {'Matched SMD':>13s}")
print("-" * 50)
for feat in features:
    # Recalculate original SMD for direct comparison
    t_mean_orig = treatment[feat].mean()
    c_mean_orig = control[feat].mean()
    pooled_sd_orig = np.sqrt((treatment[feat].var() + control[feat].var()) / 2)
    smd_orig = (t_mean_orig - c_mean_orig) / pooled_sd_orig if pooled_sd_orig > 0 else 0

    t_mean_matched = treatment_matched[feat].mean()
    c_mean_matched = control_matched[feat].mean()
    pooled_sd_matched = np.sqrt((treatment_matched[feat].var() + control_matched[feat].var()) / 2)
    smd_matched = (t_mean_matched - c_mean_matched) / pooled_sd_matched if pooled_sd_matched > 0 else 0
    print(f"{feat:<22s} {smd_orig:>+14.3f} {smd_matched:>+13.3f}")

### Step 5: Compute the Matched ATE

In [ ]:
y_treated = treatment_matched['upgraded'].values
y_matched_control = control_matched['upgraded'].values

matched_ate = (y_treated - y_matched_control).mean()
print(f"PSM-matched ATT: {matched_ate*100:+.1f} pp")

ground_truth_att = df_gt[df_gt['received_offer'] == 1]['treatment_effect'].mean()
print(f"Ground Truth ATT: {ground_truth_att*100:+.1f} pp")

### Compare all three estimates

In [ ]:
print("ATE estimates:")
print("=" * 50)
print(f"  Naive ATE   (biased):         {naive_ate*100:+6.1f} pp")
print(f"  PSM ATE     (less biased):    {matched_ate*100:+6.1f} pp")
print(f"  True ATE    (ground truth):   {true_ate*100:+6.1f} pp")
print("=" * 50)
print(f"\n  Naive bias:   {(naive_ate - true_ate)*100:+.1f} pp")
print(f"  PSM bias:     {(matched_ate - true_ate)*100:+.1f} pp")
print(f"\n  Bias reduction: {(1 - abs(matched_ate - true_ate) / abs(naive_ate - true_ate))*100:.0f}%")

## **Uplift Modeling**

### Build a T-learner model

We will use a Random Forest as the base model for $M_0$ and $M_1$

In [ ]:
# Combine the matched samples into a single dataframe
df = pd.concat([treatment_matched, control_matched], axis=0)

# Drop the 'propensity' column
df_encoded = df.drop('propensity', axis=1)

# Define feature columns (exclude ID, outcome, treatment, and ground truth)
exclude_cols = [
    'customer_id', 'current_plan', 'upgraded', 'received_offer',
    'p_upgrade_no_offer', 'p_upgrade_with_offer', 'treatment_effect',
    'ev_no_offer', 'ev_with_offer', 'optimal_action'
]
feature_cols = [c for c in df_encoded.columns if c not in exclude_cols]

X = df_encoded[feature_cols]
y = df_encoded['upgraded']
T = df_encoded['received_offer']

print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols[:10]}...")

Let's now split the data by treatment status

In [ ]:
# Control group (did NOT receive offer)
X_control = X[T == 0]
y_control = y[T == 0]

# Treatment group (received offer)
X_treatment = X[T == 1]
y_treatment = y[T == 1]

print(f"Control group: {len(X_control):,} customers")
print(f"  Upgrade rate: {y_control.mean():.3f}")

print(f"\nTreatment group: {len(X_treatment):,} customers")
print(f"  Upgrade rate: {y_treatment.mean():.3f}")

Train $M_0$ on control group using a Random Forest classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train M_0 on control group only
model_0 = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
model_0.fit(X_control, y_control)
print("Model M_0 (control) trained with XGBoost")

# Check training accuracy
train_acc_0 = model_0.score(X_control, y_control)
print(f"  Training accuracy: {train_acc_0:.3f}")

Train $M_1$ on treatment group using a Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train M_1 on treatment group only
model_1 = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
model_1.fit(X_treatment, y_treatment)
print("Model M_1 (treatment) trained with XGBoost")

# Check training accuracy
train_acc_1 = model_1.score(X_treatment, y_treatment)
print(f"  Training accuracy: {train_acc_1:.3f}")

Predict $p_0$ and $p_1$ for each customer

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Predict p_0(x) = P(upgrade | x, no offer) for all customers
df['p0_predicted'] = model_0.predict_proba(X)[:, 1]

# Predict p_1(x) = P(upgrade | x, with offer) for all customers
df['p1_predicted'] = model_1.predict_proba(X)[:, 1]

# Calculate predicted treatment effect
df['tau_predicted'] = df['p1_predicted'] - df['p0_predicted']

print(f"Average predicted p_0: {df['p0_predicted'].mean():.3f}")
print(f"Average predicted p_1: {df['p1_predicted'].mean():.3f}")
print(f"Average predicted tau: {df['tau_predicted'].mean():.3f}")

print(f"\nDistribution of predicted tau:")
print(df['tau_predicted'].describe())



Let's validate the predictions against the ground truth

In [ ]:
# combine the matched dataframe with the ground truth
df = pd.merge(df, df_gt)
df.head()

In [ ]:
mse_p0 = mean_squared_error(df['p_upgrade_no_offer'], df['p0_predicted'])
mse_p1 = mean_squared_error(df['p_upgrade_with_offer'], df['p1_predicted'])
mse_tau = mean_squared_error(df['treatment_effect'], df['tau_predicted'])

corr_p0 = np.corrcoef(df['p_upgrade_no_offer'], df['p0_predicted'])[0, 1]
corr_p1 = np.corrcoef(df['p_upgrade_with_offer'], df['p1_predicted'])[0, 1]
corr_tau = np.corrcoef(df['treatment_effect'], df['tau_predicted'])[0, 1]

print(f"\nT-learner MSE for p0:   {mse_p0:.4f}")
print(f"T-learner MSE for p1:   {mse_p1:.4f}")
print(f"T-learner MSE for tau:  {mse_tau:.4f}")

print(f"\nT-learner Correlation for p0:   {corr_p0:.3f}")
print(f"T-learner Correlation for p1:   {corr_p1:.3f}")
print(f"T-learner Correlation for tau:  {corr_tau:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# p_0 scatter
axes[0].scatter(df['p_upgrade_no_offer'], df['p0_predicted'], alpha=0.1, s=5)
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
axes[0].set_xlabel('True p_0')
axes[0].set_ylabel('Predicted p_0')
axes[0].set_title(f'p_0 corr: {corr_p0:.3f}')
axes[0].legend()

# p_1 scatter
axes[1].scatter(df['p_upgrade_with_offer'], df['p1_predicted'], alpha=0.1, s=5, color='orange')
axes[1].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
axes[1].set_xlabel('True p_1')
axes[1].set_ylabel('Predicted p_1')
axes[1].set_title(f'p_1 corr: {corr_p1:.3f}')
axes[1].legend()

# tau scatter
axes[2].scatter(df['treatment_effect'], df['tau_predicted'], alpha=0.1, s=5, color='green')
axes[2].plot([-0.1, 0.6], [-0.1, 0.6], 'r--', label='Perfect prediction')
axes[2].set_xlabel('True tau')
axes[2].set_ylabel('Predicted tau')
axes[2].set_title(f'Treatment Effect corr: {corr_tau:.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()

# Added this comment to trigger re-execution and ensure df is updated for plotting.

### Build an S-learner model

In [ ]:
# Prepare data for S-learner: combine features X and treatment indicator T
X_s = X.copy()
X_s['received_offer'] = T

# Ensure all columns are numeric (one-hot encoded treatment indicator)
X_s = pd.get_dummies(X_s, columns=['received_offer'], drop_first=False)

print(f"S-learner features shape: {X_s.shape}")
print(f"S-learner target shape: {y.shape}")

Train the S-learner model using a Random Forest Classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_s = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
model_s.fit(X_s, y)

print("S-learner model trained with Random Forest")

# Check training accuracy
train_acc_s = model_s.score(X_s, y)
print(f"  Training accuracy: {train_acc_s:.3f}")

Predict $p_0$ and $p_1$ for each customer using the S-learner model.

In [ ]:
# Create counterfactual data for prediction
# To predict p_0(x): Assume all customers received_offer = 0
X_s_control_cf = X_s.copy()
X_s_control_cf['received_offer_0'] = 1 # Ensure this column exists and is 1
X_s_control_cf['received_offer_1'] = 0 # Ensure this column exists and is 0

# To predict p_1(x): Assume all customers received_offer = 1
X_s_treatment_cf = X_s.copy()
X_s_treatment_cf['received_offer_1'] = 1 # Ensure this column exists and is 1
X_s_treatment_cf['received_offer_0'] = 0 # Ensure this column exists and is 0

# Predict p_0(x) = P(upgrade | x, no offer) for all customers
df['p0_predicted_s'] = model_s.predict_proba(X_s_control_cf)[:, 1]

# Predict p_1(x) = P(upgrade | x, with offer) for all customers
df['p1_predicted_s'] = model_s.predict_proba(X_s_treatment_cf)[:, 1]

# Calculate predicted treatment effect
df['tau_predicted_s'] = df['p1_predicted_s'] - df['p0_predicted_s']

print(f"Average predicted p_0 (S-learner): {df['p0_predicted_s'].mean():.3f}")
print(f"Average predicted p_1 (S-learner): {df['p1_predicted_s'].mean():.3f}")
print(f"Average predicted tau (S-learner): {df['tau_predicted_s'].mean():.3f}")

print(f"\nDistribution of predicted tau (S-learner):")
print(df['tau_predicted_s'].describe())

Let's validate the S-learner predictions against the ground truth using MSE and correlation.

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

mse_p0_s = mean_squared_error(df['p_upgrade_no_offer'], df['p0_predicted_s'])
mse_p1_s = mean_squared_error(df['p_upgrade_with_offer'], df['p1_predicted_s'])
mse_tau_s = mean_squared_error(df['treatment_effect'], df['tau_predicted_s'])

corr_p0_s = np.corrcoef(df['p_upgrade_no_offer'], df['p0_predicted_s'])[0, 1]
corr_p1_s = np.corrcoef(df['p_upgrade_with_offer'], df['p1_predicted_s'])[0, 1]
corr_tau_s = np.corrcoef(df['treatment_effect'], df['tau_predicted_s'])[0, 1]

print(f"S-learner MSE for p0:   {mse_p0_s:.4f}")
print(f"S-learner MSE for p1:   {mse_p1_s:.4f}")
print(f"S-learner MSE for tau:  {mse_tau_s:.4f}")

print(f"\nS-learner Correlation for p0:   {corr_p0_s:.3f}")
print(f"S-learner Correlation for p1:   {corr_p1_s:.3f}")
print(f"S-learner Correlation for tau:  {corr_tau_s:.3f}")

print("\n--- Comparison with T-learner ---")
print(f"T-learner MSE for p0:   {mse_p0:.4f}")
print(f"T-learner MSE for p1:   {mse_p1:.4f}")
print(f"T-learner MSE for tau:  {mse_tau:.4f}")

print(f"\nT-learner Correlation for p0:   {corr_p0:.3f}")
print(f"T-learner Correlation for p1:   {corr_p1:.3f}")
print(f"T-learner Correlation for tau:  {corr_tau:.3f}")

Visualize the S-learner predictions against the ground truth.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# p_0 scatter
axes[0].scatter(df['p_upgrade_no_offer'], df['p0_predicted_s'], alpha=0.1, s=5)
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
axes[0].set_xlabel('True p_0')
axes[0].set_ylabel('Predicted p_0 (S-learner)')
axes[0].set_title(f'p_0 corr: {corr_p0_s:.3f}')
axes[0].legend()

# p_1 scatter
axes[1].scatter(df['p_upgrade_with_offer'], df['p1_predicted_s'], alpha=0.1, s=5, color='orange')
axes[1].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
axes[1].set_xlabel('True p_1')
axes[1].set_ylabel('Predicted p_1 (S-learner)')
axes[1].set_title(f'p_1 corr: {corr_p1_s:.3f}')
axes[1].legend()

# tau scatter
axes[2].scatter(df['treatment_effect'], df['tau_predicted_s'], alpha=0.1, s=5, color='green')
axes[2].plot([-0.1, 0.6], [-0.1, 0.6], 'r--', label='Perfect prediction')
axes[2].set_xlabel('True tau')
axes[2].set_ylabel('Predicted tau (S-learner)')
axes[2].set_title(f'Treatment Effect corr: {corr_tau_s:.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()

### Perturbation-based Feature Importance for S-learner

Let's calculate the permutation importance for the S-learner. This method assesses feature importance by measuring the increase in the CATE prediction error when the values of a single feature are randomly shuffled. A larger increase in error indicates a more important feature.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Get baseline MSE for CATE from the S-learner (OHE)
baseline_mse_tau_s = mse_tau_s # This was calculated previously in cell 59b0c2d6

# 2. Get the full counterfactual dataframes for all customers
#    These were created when predicting for the S-learner in cell ca2782ab
#    X_s_control_cf (all received_offer=0)
#    X_s_treatment_cf (all received_offer=1)

# List of features to perturb (these are the original features from X)
features_to_perturb = feature_cols

permutation_importances = {}

for feature in features_to_perturb:
    # Create copies of the counterfactual dataframes for perturbation
    X_control_perturbed = X_s_control_cf.copy()
    X_treatment_perturbed = X_s_treatment_cf.copy()

    # Shuffle the current feature column
    # Ensure the feature exists in the dataframe before attempting to shuffle
    if feature in X_control_perturbed.columns:
        X_control_perturbed[feature] = np.random.permutation(X_control_perturbed[feature])
        X_treatment_perturbed[feature] = np.random.permutation(X_treatment_perturbed[feature])
    else:
        print(f"Warning: Feature '{feature}' not found in X_s_control_cf. Skipping.")
        continue

    # Predict p0 and p1 with the perturbed feature
    p0_perturbed = model_s.predict_proba(X_control_perturbed)[:, 1]
    p1_perturbed = model_s.predict_proba(X_treatment_perturbed)[:, 1]

    # Calculate perturbed tau
    tau_perturbed = p1_perturbed - p0_perturbed

    # Calculate MSE of the perturbed tau against the true treatment effect
    mse_tau_perturbed = mean_squared_error(df['treatment_effect'], tau_perturbed)

    # Importance is the increase in MSE
    importance = mse_tau_perturbed - baseline_mse_tau_s
    permutation_importances[feature] = importance

# Convert to DataFrame for easier plotting
perm_importance_df = pd.DataFrame(
    list(permutation_importances.items()), columns=['Feature', 'Importance']
).sort_values(by='Importance', ascending=False)

print("Permutation-based Feature Importances for S-learner (CATE):")
print(perm_importance_df.head(10))

# Plotting
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=perm_importance_df.head(15), palette='viridis', legend=False)
plt.title('Permutation-based Feature Importances for S-learner (CATE)')
plt.xlabel('Increase in MSE of CATE Prediction')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Partial Dependence Plots (PDPs) for S-learner

Let's generate Partial Dependence Plots (PDPs) to understand the marginal effect of individual features on the S-learner's predicted CATE. A PDP shows the relationship between a feature (or a few features) and the predicted outcome (in our case, the CATE), marginalizing over the values of all other features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_pdp(model, X_df, feature_name, n_points=20, ax=None):
    """
    Generates a Partial Dependence Plot (PDP) for a given feature on the CATE predicted by an S-learner model.

    Args:
        model: The trained S-learner model.
        X_df (pd.DataFrame): The original feature DataFrame (before adding treatment indicators).
        feature_name (str): The name of the feature to plot PDP for.
        n_points (int): Number of points to evaluate for numerical features.
        ax (matplotlib.axes.Axes, optional): Axes to plot on. If None, a new figure and axes are created.
    """
    # Create a base instance using mean for numerical and mode for categorical features
    base_row_dict = {}
    for col in X_df.columns:
        if X_df[col].dtype in ['int64', 'float64']:
            base_row_dict[col] = X_df[col].mean()
        else:
            # For boolean or other categoricals, use mode
            base_row_dict[col] = X_df[col].mode()[0]

    # Prepare the template for the model's input (original features + received_offer_0 + received_offer_1)
    X_s_template_cols = list(X_df.columns) + ['received_offer_0', 'received_offer_1']
    X_s_base_df = pd.DataFrame(columns=X_s_template_cols)
    X_s_base_df.loc[0] = 0 # Initialize with zeros to ensure all columns exist and are numeric where expected

    # Fill the base_row_dict values into the template dataframe
    for col, value in base_row_dict.items():
        if col in X_s_base_df.columns:
            X_s_base_df[col] = value

    # Determine feature values to evaluate
    if X_df[feature_name].nunique() <= 5 and X_df[feature_name].dtype != 'float64': # Treat low-cardinality non-float as categorical
        feature_values = sorted(X_df[feature_name].unique())
    else: # Numerical features
        feature_values = np.linspace(X_df[feature_name].min(), X_df[feature_name].max(), n_points)

    cate_values = []

    for val in feature_values:
        # Control counterfactual: received_offer=0
        X_control_cf = X_s_base_df.copy()
        X_control_cf['received_offer_0'] = 1
        X_control_cf['received_offer_1'] = 0
        X_control_cf[feature_name] = val

        # Treatment counterfactual: received_offer=1
        X_treatment_cf = X_s_base_df.copy()
        X_treatment_cf['received_offer_0'] = 0
        X_treatment_cf['received_offer_1'] = 1
        X_treatment_cf[feature_name] = val

        # Predict p0 and p1
        p0 = model.predict_proba(X_control_cf)[:, 1]
        p1 = model.predict_proba(X_treatment_cf)[:, 1]

        # Calculate CATE
        cate = p1 - p0
        cate_values.append(cate[0])

    # Plotting
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(feature_values, cate_values, marker='o', linestyle='-', markersize=4)
    ax.set_title(f'PDP for {feature_name}')
    ax.set_xlabel(feature_name)
    ax.set_ylabel('CATE (P(upgrade|T=1) - P(upgrade|T=0))')
    ax.grid(True)

    return ax

In [ ]:
# List of features for which to generate PDPs
features_for_pdp = [
    'data_usage_gb',
    'late_payments_12mo',
    'age',
    'tenure_months',
    'payment_issues',
    'minutes_used'
]

# Determine grid size for subplots
n_features = len(features_for_pdp)
n_cols = 3 # Number of columns in the plot grid
n_rows = (n_features + n_cols - 1) // n_cols # Calculate number of rows needed

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, feature in enumerate(features_for_pdp):
    plot_pdp(model_s, X, feature, ax=axes[i])

# Remove any unused subplots if n_features is not a perfect multiple of n_cols
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to make space for suptitle
plt.suptitle('Partial Dependence Plots (PDPs) for S-learner CATE', fontsize=16)
plt.show()

## Uplift-Driven Targeting vs Oracle

Put the whole pipeline together: use an **S-learner** to estimate per-customer treatment effects τ̂(x), select the top 2,500 customers by predicted uplift, and compare the resulting profit to:

1. **Oracle**: top 2,500 by TRUE τ (the best possible targeting)
2. **Random**: 2,500 random customers
3. **No offers**: baseline

This closes the loop between Week 14 (estimation) and Week 13 (optimization).

First, let's set up the profit evaluation function

In [ ]:
# Ground truth values
p0_true = df['p_upgrade_no_offer'].values
p1_true = df['p_upgrade_with_offer'].values
tau_true = df['treatment_effect'].values

# Prediction by S-learner
tau_hat = df['tau_predicted_s'].values

REVENUE = 480  # per upgrade
COST = 50      # per offer
BUDGET = 2500  # offer limit

def evaluate_strategy(send_mask):
    """Given a boolean mask of which customers to offer,
       compute total expected profit using ground-truth p0, p1."""
    profit_per_customer = np.where(
        send_mask,
        p1_true * REVENUE - COST,  # if offered: expected revenue minus cost
        p0_true * REVENUE           # if not offered: expected revenue, no cost
    )
    return profit_per_customer.sum()

print(f"Budget: {BUDGET:,} offers")
print(f"Break-even threshold: tau > {COST/REVENUE:.4f}")

In [ ]:
# --- Strategy 1: S-learner — top 2,500 by predicted tau ---
slearner_indices = np.argsort(-tau_hat)[:BUDGET]
slearner_mask = np.zeros(len(X), dtype=bool)
slearner_mask[slearner_indices] = True

# --- Strategy 2: Oracle — top 2,500 by TRUE tau (best possible) ---
oracle_indices = np.argsort(-tau_true)[:BUDGET]
oracle_mask = np.zeros(len(X), dtype=bool)
oracle_mask[oracle_indices] = True

# --- Strategy 3: Random 2,500 ---
np.random.seed(42)
random_indices = np.random.choice(len(X), BUDGET, replace=False)
random_mask = np.zeros(len(X), dtype=bool)
random_mask[random_indices] = True

# --- Strategy 4: No offers baseline ---
no_offer_mask = np.zeros(len(X), dtype=bool)

# Evaluate each
profit_none      = evaluate_strategy(no_offer_mask)
profit_random    = evaluate_strategy(random_mask)
profit_slearner  = evaluate_strategy(slearner_mask)
profit_oracle    = evaluate_strategy(oracle_mask)

print(f"Total expected profit under each strategy:")
print(f"=" * 60)
print(f"  No offers (baseline):            ${profit_none:>12,.0f}")
print(f"  Random 2,500 offers:             ${profit_random:>12,.0f}  (+${profit_random-profit_none:,.0f})")
print(f"  S-learner (top 2,500 by tau):    ${profit_slearner:>12,.0f}  (+${profit_slearner-profit_none:,.0f})")
print(f"  Oracle (top 2,500 by TRUE tau):  ${profit_oracle:>12,.0f}  (+${profit_oracle-profit_none:,.0f})")

How much of the oracle's value did we capture?

In [ ]:
oracle_gain   = profit_oracle   - profit_none
slearner_gain = profit_slearner - profit_none
random_gain   = profit_random   - profit_none

print(f"Incremental profit over no-offers baseline:")
print(f"  Random:     ${random_gain:>10,.0f}  ({100 * random_gain / oracle_gain:5.1f}% of oracle)")
print(f"  S-learner:  ${slearner_gain:>10,.0f}  ({100 * slearner_gain / oracle_gain:5.1f}% of oracle)")
print(f"  Oracle:     ${oracle_gain:>10,.0f}  (100.0%)")
print()
print(f"S-learner gap to oracle:             ${profit_slearner - profit_oracle:+,.0f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Absolute profit comparison
strategies = ['No offers', 'Random\n2,500', 'S-learner\n(top \u03c4\u0302)', 'Oracle\n(top \u03c4 true)']
profits = [profit_none, profit_random, profit_slearner, profit_oracle]
colors = ['lightgray', '#C19A2B', '#990011', '#1E2761', '#2C5F2D']

bars = ax1.bar(strategies, [p/1000 for p in profits], color=colors, edgecolor='white', linewidth=1.5)
for bar, p in zip(bars, profits):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
             f'${p/1000:.0f}K', ha='center', fontsize=11, fontweight='bold')

ax1.set_ylabel('Expected profit ($ thousands)', fontsize=11)
ax1.set_title('Total expected profit by targeting strategy', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(min(profits)/1000 * 0.97, max(profits)/1000 * 1.05)

# Panel 2: % of oracle gain captured
fractions = [
    (random_gain / oracle_gain) * 100,
    (slearner_gain / oracle_gain) * 100,
    100.0
]
labels = ['Random', 'S-learner\n(top \u03c4\u0302)', 'Oracle']
colors2 = ['#C19A2B', '#990011', '#1E2761', '#2C5F2D']

bars2 = ax2.bar(labels, fractions, color=colors2, edgecolor='white', linewidth=1.5)
for bar, f in zip(bars2, fractions):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{f:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax2.set_ylabel('% of oracle incremental profit captured', fontsize=11)
ax2.set_title('How much of the oracle\'s value did we capture?', fontsize=12, fontweight='bold')
ax2.axhline(100, color='#2C5F2D', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 115)

plt.tight_layout()
plt.show()